# Python Refresh for ML

This notebook does not reteach Python from scratch. It reviews the Python ideas that show up again and again when you read PyTorch examples, write training scripts, or implement small datasets.

The main thread is this: a training project is mostly functions and classes passing data around. Function signatures control how configuration values enter your code. Classes keep related state and behavior together. Special methods such as `__len__` and `__getitem__` let your own objects behave like standard Python containers, which is exactly what PyTorch expects from a `Dataset`.

Use this notebook actively. Read each example first, predict what it should print, then run it. For each exercise, implement the TODOs before checking the reference solution.

## Learning Goals

After finishing this notebook, you should be able to read function signatures used in training scripts and understand why some arguments have defaults while others must be passed by name. You should also understand the Python class mechanics behind `nn.Module` and `Dataset`: what `__init__` stores, what instance attributes are, why subclasses call `super().__init__()`, and how methods use the state stored on `self`.

By the end, you should be able to write simple classes with `__len__` and `__getitem__`, and you should be able to add basic validation with `assert`, `ValueError`, and `KeyError` so bugs fail near their source instead of much later in the training loop.

## Function Arguments

Function arguments are how configuration enters machine learning code. A training function might need a learning rate, a batch size, a random seed, an output directory, and several optional switches. If the function signature is clear, the call site is readable. If the signature is loose or ambiguous, it becomes easy to swap values accidentally.

In the example below, `model_name` is required because the function cannot build a useful experiment name without it. `lr` and `batch_size` have defaults because there are reasonable fallback values. `seed` and `extra_tags` are keyword-only because these values are easier to understand when the caller writes their names explicitly.

In [ ]:
def build_experiment_name(model_name, lr=1e-3, batch_size=32, *, seed=42, extra_tags=None):
    if extra_tags is None:
        extra_tags = []

    tag_text = "-".join(extra_tags) if extra_tags else "base"
    return f"{model_name}_lr{lr}_bs{batch_size}_seed{seed}_{tag_text}"


name = build_experiment_name(
    "mlp",
    batch_size=64,
    seed=7,
    extra_tags=["aug", "dropout"],
)

print("experiment name / experiment name:", name)

# Think:
# Why must seed be passed as seed=7 after the * marker?


The default values `lr=1e-3` and `batch_size=32` are used only when the caller does not provide those arguments. This lets a function have sensible behavior while still allowing the caller to override the defaults.

The bare `*` in the function signature is not an argument by itself. It is a separator. Everything after it must be passed by keyword, so `seed` must be written as `seed=7` instead of being passed as another unnamed positional value. This is useful because a call like `build_experiment_name("mlp", 0.01, 128, 7)` is hard to read: the final `7` could mean many things. A call that says `seed=7` is self-documenting.

The `extra_tags=None` pattern avoids a common Python bug. If the default were `extra_tags=[]`, that same list object would be reused across calls. Using `None` lets the function create a fresh list only when it needs one.

In [ ]:
# Exercise 1
#
# Implement summarize_split().
#
# This function describes a dataset split. The caller gives you the number of
# training samples, validation samples, and optionally test samples. The function
# should compute total_size and return all settings in a dictionary.
#
# What each argument means:
# - train_size: number of samples in the training split
# - val_size: number of samples in the validation split
# - test_size: number of samples in the test split; it defaults to 0
# - shuffle: whether the data should be shuffled before splitting
# - stratify: whether class proportions should be preserved across splits
#
# Important signature requirement:
# shuffle and stratify come after *, so callers must write shuffle=True and
# stratify=True. They should not be passable as unnamed positional arguments.
#
# Expected behavior:
# summarize_split(800, 100, test_size=100, shuffle=True, stratify=True)
# should return:
# {
#     "train_size": 800,
#     "val_size": 100,
#     "test_size": 100,
#     "total_size": 1000,
#     "shuffle": True,
#     "stratify": True,
# }

def summarize_split(train_size, val_size, test_size=0, *, shuffle=True, stratify=False):
    # write your implementation here
    pass


# print(summarize_split(800, 100, test_size=100, shuffle=True, stratify=True))

In [ ]:
# Exercise 1 Reference Solution

def summarize_split_solution(train_size, val_size, test_size=0, *, shuffle=True, stratify=False):
    total_size = train_size + val_size + test_size
    return {
        "train_size": train_size,
        "val_size": val_size,
        "test_size": test_size,
        "total_size": total_size,
        "shuffle": shuffle,
        "stratify": stratify,
    }


print(summarize_split_solution(800, 100, test_size=100, shuffle=True, stratify=True))

### `*args` and `**kwargs`

`*args` and `**kwargs` are used when a function needs to accept a flexible number of arguments. They are not magic names, but these two names are conventional and you should recognize them when reading library code.

`*args` collects extra positional arguments into a tuple. In the call `log_metrics(3, 0.91, 0.27, loss=0.27)`, the first value `3` is assigned to `epoch`, and the extra unnamed values `0.91` and `0.27` are collected into `metric_values`.

`**kwargs` collects extra keyword arguments into a dictionary. In that same call, `loss=0.27` is collected into `named_metrics` as `{"loss": 0.27}`.

This pattern is common in ML wrapper functions. For example, a helper might accept `**loader_kwargs` and pass those values into `DataLoader`. That lets the caller provide options such as `batch_size=32`, `shuffle=True`, or `num_workers=2` without the helper listing every possible DataLoader option.

You should not use `*args` and `**kwargs` just to avoid writing a clear signature. For beginner project code, explicit parameters are usually easier to understand and debug.

In [ ]:
def log_metrics(epoch, *metric_values, **named_metrics):
    print(f"epoch={epoch}")
    print("positional metrics / positional metrics:", metric_values)
    print("named metrics / named metrics:", named_metrics)


log_metrics(3, 0.91, 0.27, loss=0.27, accuracy=0.91)

## Classes, Inheritance, and `super()`

PyTorch model and dataset code relies heavily on classes. A class is useful when some data and some behavior belong together. For example, a metric tracker needs to remember previous values, and it also needs methods for updating those values and computing a result.

`__init__` is the method that prepares a new object. Attributes such as `self.name` and `self.values` are stored on that specific object, so later methods can use them. When you write `loss_tracker.update(0.95)`, the method reads and changes the state stored inside `loss_tracker`.

Inheritance means one class can reuse behavior from another class. In the example below, `RunningAverage` is a more specific kind of `MetricTracker`. It reuses the parent class's initialization of `name` and `values`, then adds its own state, `total` and `count`. The call to `super().__init__(name)` is what runs the parent initialization before the child adds its extra fields.

In [ ]:
class MetricTracker:
    def __init__(self, name):
        self.name = name
        self.values = []

    def update(self, value):
        self.values.append(float(value))

    def compute(self):
        if not self.values:
            return 0.0
        return sum(self.values) / len(self.values)


class RunningAverage(MetricTracker):
    def __init__(self, name):
        super().__init__(name)
        self.total = 0.0
        self.count = 0

    def update(self, value):
        value = float(value)
        self.total += value
        self.count += 1
        self.values.append(value)

    def compute(self):
        if self.count == 0:
            return 0.0
        return self.total / self.count


loss_tracker = RunningAverage("train_loss")
for loss in [0.95, 0.72, 0.51]:
    loss_tracker.update(loss)

print("name / name:", loss_tracker.name)
print("average / average:", loss_tracker.compute())

In this example, shared logic means the setup that both `MetricTracker` and `RunningAverage` need. The parent class `MetricTracker` defines the basic idea that every tracker has a `name` and a list called `values`. The child class `RunningAverage` does not need to rewrite that setup, because `super().__init__(name)` asks the parent class to do it.

`RunningAverage` then extends the parent behavior by adding `total` and `count`. It also overrides `update` and `compute` because a running average can be computed more directly from those two numbers. This is the same pattern you will use later with `nn.Module`: PyTorch's parent class sets up the machinery for parameters, devices, and submodules, and your child class adds the layers and the `forward` method for your specific model.

In [ ]:
# Exercise 2
#
# Implement BatchCounter.
#
# A BatchCounter object should keep track of batches that have already been
# seen. In this exercise, a "batch" can be any Python object with a length, such
# as a list. For example, [1, 2, 3, 4] is one batch with 4 samples.
#
# __init__ responsibilities:
# - Create self.num_batches and set it to 0.
# - Create self.num_samples and set it to 0.
# These attributes store the state that later methods will update.
#
# update(batch) responsibilities:
# - This method is called once per batch.
# - Increase self.num_batches by 1.
# - Increase self.num_samples by len(batch).
# - It does not need to return anything; its job is to update the object's state.
#
# summary() responsibilities:
# - Compute avg_batch_size as num_samples / num_batches.
# - If no batches have been seen yet, use 0.0 to avoid dividing by zero.
# - Return a dict with num_batches, num_samples, and avg_batch_size.
#
# Expected behavior:
# counter = BatchCounter()
# counter.update([1, 2, 3, 4])
# counter.update([5, 6])
# print(counter.summary())
# should produce a dict equivalent to:
# {"num_batches": 2, "num_samples": 6, "avg_batch_size": 3.0}

class BatchCounter:
    def __init__(self):
        # TODO
        pass

    def update(self, batch):
        # TODO
        pass

    def summary(self):
        # TODO
        pass


# counter = BatchCounter()
# counter.update([1, 2, 3, 4])
# counter.update([5, 6])
# print(counter.summary())

In [ ]:
# Exercise 2 Reference Solution

class BatchCounterSolution:
    def __init__(self):
        self.num_batches = 0
        self.num_samples = 0

    def update(self, batch):
        self.num_batches += 1
        self.num_samples += len(batch)

    def summary(self):
        avg_batch_size = 0.0 if self.num_batches == 0 else self.num_samples / self.num_batches
        return {
            "num_batches": self.num_batches,
            "num_samples": self.num_samples,
            "avg_batch_size": avg_batch_size,
        }


counter = BatchCounterSolution()
counter.update([1, 2, 3, 4])
counter.update([5, 6])
print(counter.summary())

## `__len__` and `__getitem__`

`__len__` and `__getitem__` are special methods that let your object behave like a standard Python container. When Python sees `len(dataset)`, it calls `dataset.__len__()`. When Python sees `dataset[1]`, it calls `dataset.__getitem__(1)`.

This matters for PyTorch because a custom `Dataset` follows the same idea. A DataLoader needs to know how many samples exist, and it needs to ask for one sample by index. If your class answers those two questions correctly, PyTorch can batch and shuffle your data for you.

In the example below, `ToyDataset` stores all samples in `self.samples`. Its `__len__` method returns the number of stored samples, and its `__getitem__` method returns one indexed sample in a consistent format.

In [ ]:
class ToyDataset:
    def __init__(self, samples):
        self.samples = list(samples)

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, index):
        features, label = self.samples[index]
        return {"x": features, "y": label}


samples = [
    ([0.1, 0.2], 0),
    ([0.7, 0.9], 1),
    ([0.3, 0.4], 0),
]

dataset = ToyDataset(samples)
print("number of samples / dataset length:", len(dataset))
print("item 1 / item 1:", dataset[1])

In [ ]:
# Exercise 3
#
# Implement WindowDataset.
#
# This dataset turns a single list of values into many input-target pairs.
# With values = [10, 11, 12, 13, 14] and window_size = 2:
# - index 0 should return {"x": [10, 11], "y": 12}
# - index 1 should return {"x": [11, 12], "y": 13}
# - index 2 should return {"x": [12, 13], "y": 14}
#
# __len__ responsibilities:
# - Return the number of valid windows.
# - The last possible target is the final value in self.values.
# - For 5 values and window_size 2, there are 3 valid windows.
#
# __getitem__(index) responsibilities:
# - Build the input window from self.values[index : index + self.window_size].
# - Build the target from self.values[index + self.window_size].
# - Return a dict with keys "x" and "y".

class WindowDataset:
    def __init__(self, values, window_size):
        self.values = list(values)
        self.window_size = window_size

    def __len__(self):
        # TODO
        pass

    def __getitem__(self, index):
        # TODO
        pass


# ds = WindowDataset([10, 11, 12, 13, 14], window_size=2)
# print(len(ds))
# print(ds[0])
# print(ds[1])
# print(ds[2])

In [ ]:
# Exercise 3 Reference Solution

class WindowDatasetSolution:
    def __init__(self, values, window_size):
        self.values = list(values)
        self.window_size = window_size

    def __len__(self):
        return len(self.values) - self.window_size

    def __getitem__(self, index):
        window = self.values[index : index + self.window_size]
        target = self.values[index + self.window_size]
        return {"x": window, "y": target}


ds = WindowDatasetSolution([10, 11, 12, 13, 14], window_size=2)
print("length / length:", len(ds))
print(ds[0])
print(ds[1])
print(ds[2])

## Context Managers, Assertions, and Exceptions

Training code often fails for simple reasons: a file was not closed, a batch has mismatched feature and label lengths, or a required key is missing from a row. Context managers, assertions, and exceptions give you controlled ways to handle those cases.

A context manager is the pattern behind `with ... as ...`. It sets something up, lets you use it, and then cleans it up when the block ends. File handling commonly uses this pattern.

An `assert` is a quick sanity check for a condition that should always be true while developing. If the condition is false, Python raises an error immediately. For user-facing validation, explicit exceptions such as `ValueError` and `KeyError` are often clearer because they describe exactly what went wrong.

In [ ]:
from io import StringIO


with StringIO("epoch=1,loss=0.82\nepoch=2,loss=0.64\n") as f:
    lines = [line.strip() for line in f if line.strip()]

print("log lines / log lines:", lines)


def validate_batch(features, labels):
    assert len(features) == len(labels), "features and labels must have the same length"
    if len(features) == 0:
        raise ValueError("batch must not be empty / batch must not be empty")
    return True


print("validation result / validation result:", validate_batch([[1, 2], [3, 4]], [0, 1]))

In [ ]:
# Exercise 4
# Debugging task.
#
# The class below is intended to behave like a small dataset:
# - len(ds) should return the number of rows.
# - ds[0] should return the first row.
#
# It has at least two bugs.
#
# What to check:
# - Python calls __len__, not len, when you write len(ds).
# - __getitem__ should read from the attribute that was created in __init__.
#
# Your task:
# - Create a fixed version of the class.
# - Instantiate it with [("a", 0), ("b", 1), ("c", 0)].
# - Verify that len(fixed_ds) is 3.
# - Verify that fixed_ds[0] is ("a", 0).

class BrokenDataset:
    def __init__(self, rows):
        self.rows = rows

    def len(self):
        return len(self.rows)

    def __getitem__(self, index):
        return self.row[index]


# TODO:
# create an instance and verify len(ds) and ds[0]

In [ ]:
# Exercise 4 Reference Solution

class FixedDataset:
    def __init__(self, rows):
        self.rows = rows

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, index):
        return self.rows[index]


fixed_ds = FixedDataset([("a", 0), ("b", 1), ("c", 0)])
print("length / length:", len(fixed_ds))
print("first sample / first sample:", fixed_ds[0])

## Integrated Mini Exercise

This final exercise combines the pieces from the notebook into a tiny dataset-like class. The class receives a list of dictionaries, where each dictionary is one row. Some keys are features, and one key is the target.

The point is not to build a full PyTorch Dataset yet. The point is to practice the same responsibilities in plain Python: store constructor arguments on `self`, report the number of samples, return one sample by index, and raise a clear error if required data is missing.

In [ ]:
# Exercise 5
#
# Implement MiniTabularDataset.
#
# records is a list of dictionaries. Each dictionary is one row:
# {"hours": 1.5, "attendance": 0.70, "passed": 0}
#
# feature_keys tells the dataset which values should become the feature list.
# For feature_keys=["hours", "attendance"], the first row's features should be
# [1.5, 0.70]. The order must match feature_keys exactly.
#
# target_key tells the dataset which value should become the target. For
# target_key="passed", the first row's target should be 0.
#
# __len__ responsibilities:
# - Return the number of records.
#
# __getitem__(index) responsibilities:
# - Read the row at self.records[index].
# - If self.target_key is missing from that row, raise KeyError.
# - Build features by reading each key in self.feature_keys, in order.
# - Read target from self.target_key.
# - Return (features, target).
#
# Expected behavior:
# ds = MiniTabularDataset(records, feature_keys=["hours", "attendance"], target_key="passed")
# len(ds) should be 3.
# ds[0] should be ([1.5, 0.70], 0).

records = [
    {"hours": 1.5, "attendance": 0.70, "passed": 0},
    {"hours": 3.0, "attendance": 0.90, "passed": 1},
    {"hours": 2.2, "attendance": 0.80, "passed": 1},
]


class MiniTabularDataset:
    def __init__(self, records, feature_keys, target_key):
        self.records = list(records)
        self.feature_keys = list(feature_keys)
        self.target_key = target_key

    def __len__(self):
        # TODO
        pass

    def __getitem__(self, index):
        # TODO
        pass


# ds = MiniTabularDataset(records, feature_keys=["hours", "attendance"], target_key="passed")
# print(len(ds))
# print(ds[0])

In [ ]:
# Exercise 5 Reference Solution

class MiniTabularDatasetSolution:
    def __init__(self, records, feature_keys, target_key):
        self.records = list(records)
        self.feature_keys = list(feature_keys)
        self.target_key = target_key

    def __len__(self):
        return len(self.records)

    def __getitem__(self, index):
        row = self.records[index]
        if self.target_key not in row:
            raise KeyError(f"missing target key: {self.target_key}")

        features = [row[key] for key in self.feature_keys]
        target = row[self.target_key]
        return features, target


ds = MiniTabularDatasetSolution(records, feature_keys=["hours", "attendance"], target_key="passed")
print("length / length:", len(ds))
print(ds[0])
print(ds[1])

## Summary

The main idea of this notebook is that Python mechanics become PyTorch mechanics very quickly. A training utility is just a function with a clear signature. A model or dataset is just a class that stores state in `__init__` and exposes behavior through methods. A PyTorch `Dataset` is built on the same `len(obj)` and `obj[index]` protocol you practiced here.

When you move to later notebooks, pay attention to where each piece of state lives. If a value should be remembered across method calls, it usually belongs on `self`. If a value is only needed inside one calculation, it can stay as a local variable. If a method receives data, ask whether it should update the object, return a value, or both.

Before moving on, make sure you can answer these questions in full sentences:

1. Why does `seed` have to be passed as `seed=7` in the first example?
2. What state does `BatchCounter` need to remember between calls to `update`?
3. What does `super().__init__(name)` do in `RunningAverage`?
4. Why does a dataset-like class need both `__len__` and `__getitem__`?
5. Why is raising `KeyError` better than silently returning a wrong target?

Suggested next step: move to the NumPy core notebook and build stronger shape intuition.